# Mushroom — Gradient Boosting

Eén modeltype per notebook. Oorspronkelijke Mushroom-modellen: Andrew; de gedeelde
vergelijking en deze reproduceerbare versie: Codex op verzoek van Jorre.
Jorre rondt de definitieve taakverdeling en GenAI-policy later af (stap 5).

De huidige referentie is de correct uitgevoerde vergelijking op dezelfde 4.100
trainingsrijen, 900 ontwikkeltestrecords en vijf folds. Het model wordt hieronder
werkelijk opnieuw gefit met de vastgelegde configuratie. Alle preprocessing wordt
binnen de fold geleerd. De ontwikkeltest is eerder bekeken en is geen nieuwe
onafhankelijke eindtest. Dit notebook wijzigt de website of bronstudie niet.

We laden de vastgelegde parameters en leggen uit waarom dit model is opgenomen.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'MushroomFinal/mushroom_project_dataset.csv').exists())
sys.path.insert(0, str(ROOT))
summary = json.loads((ROOT/'MushroomFinal/comparison_correct/comparison_summary.json').read_text(encoding='utf-8'))
key = 'gb_final'
display(pd.Series(summary['classifier_parameters'][key], name='Vastgelegde parameters'))


ccp_alpha                            0.0
criterion                   friedman_mse
init                                None
learning_rate                        0.1
loss                            log_loss
max_depth                              3
max_features                        None
max_leaf_nodes                      None
min_impurity_decrease                0.0
min_samples_leaf                       1
min_samples_split                      2
min_weight_fraction_leaf             0.0
n_estimators                         200
n_iter_no_change                    None
random_state                          42
subsample                            0.8
tol                               0.0001
validation_fraction                  0.1
verbose                                0
warm_start                         False
Name: Vastgelegde parameters, dtype: object

## Oorspronkelijk tuningonderzoek

Deze zoekruimte komt uit het oorspronkelijke projectnotebook. De oorspronkelijke
uitvoer blijft daar bewaard. De eerdere proximity-imputatie was vóór CV geleerd;
daarom gebruiken we die scores niet als eindbewijs. De gecorrigeerde vijf-fold-
evaluatie hieronder leert preprocessing opnieuw binnen iedere fold. Dit is een
reproductie van de vastgelegde configuratie, geen nieuwe gridsearch of modelkeuze.

```python
param_grid_gb = {
    "classifier__n_estimators": [100, 200],
    "classifier__learning_rate": [0.03, 0.05, 0.1],
    "classifier__max_depth": [2, 3],
    "classifier__min_samples_leaf": [1, 3],
    "classifier__subsample": [0.8, 1.0]
}
```

Boosting corrigeert opeenvolgend fouten. De proximity-imputatie en alle gewone preprocessing worden binnen de fold geleerd; de extra complexiteit moet met gemeten prestaties worden afgewogen.

We fitten vijf trainingsfolds en daarna één model op alle trainingsrijen. De code controleert alle 900 labels en kansen tegen de opgeslagen referentie.

In [2]:
from Final.tools.model_workflow import reproduce
result = reproduce('gb_final')
display(pd.DataFrame(result['folds']))
display(pd.Series({k:v for k,v in result.items() if k != 'folds'}))
assert result['labels_match_original'] and result['max_probability_difference'] < 1e-12


gb_final fold 1 /5


gb_final fold 2 /5


gb_final fold 3 /5


gb_final fold 4 /5


gb_final fold 5 /5


,fold,accuracy,balanced_accuracy,macro_f1,precision_p,recall_p,f1_p,roc_auc,average_precision,tn,fp,fn,tp
0,1,0.748780,0.696953,0.705614,0.769231,0.482315,0.592885,0.777409,0.696109,464,45,161,150
1,2,0.752439,0.698649,0.707636,0.787234,0.475884,0.593186,0.781395,0.725739,469,40,163,148
2,3,0.762195,0.717109,0.726842,0.767442,0.532258,0.628571,0.791844,0.740276,460,50,145,165
3,4,0.784146,0.736022,0.748422,0.830846,0.538710,0.653620,0.799551,0.766514,476,34,143,167
4,5,0.750000,0.698450,0.707363,0.766497,0.487097,0.595661,0.770183,0.705975,464,46,159,151


model                          gb_final
cv_macro_f1_mean               0.719175
cv_macro_f1_std                0.018512
cv_recall_p_mean               0.503253
accuracy                       0.784444
balanced_accuracy              0.728694
macro_f1                       0.741733
precision_p                    0.880829
recall_p                       0.498534
f1_p                           0.636704
roc_auc                        0.821607
average_precision              0.786504
tn                                  536
fp                                   23
fn                                  171
tp                                  170
fit_and_cv_seconds            13.917501
training_rows                      4100
test_rows                           900
labels_match_original              True
max_probability_difference          0.0
sklearn                           1.4.2
dtype: object

De uiteindelijke keuze staat in `09_vergelijking.ipynb`. Een hoge score voor de eetbare klasse is geen bevestiging van eetbaarheid; de website gebruikt een afzonderlijke voorzichtige beslisregel.